In [1]:
import pandas as pd
from ortools.sat.python import cp_model


pers_df = pd.read_excel('Phase2.xlsx', sheet_name='Pers')
pos_df = pd.read_excel('Phase2.xlsx', sheet_name='Pos')

In [2]:
# Mapping
citizenship_map = {"Local": 1, "PR": 2, "Foreigner": 3, "Any": 4}
field_map = {"IT":1, "MED":2, "SALES":3, "ENGINEER":4, "CHEMIST":5, "LEAD":6, "ANY":7}
tag_map = {"HR":1, "Int":2, "Ops":3, "Logs":4, "Plans":5, "Train":6, "Main":7}
tier_map = {"Junior":1, "Mid":2, "Senior":3}

pers_df["Citizenship"] = pers_df["Citizenship"].map(citizenship_map)
pers_df["Tier"] = pers_df["Tier"].map(tier_map)
pers_df["Field"] = pers_df["Field"].map(field_map)

pos_df["Req_Citizenship"] = pos_df["Req_Citizenship"].map(citizenship_map)
pos_df["Req_Tier"] = pos_df["Req_Tier"].map(tier_map)
pos_df["Req_Field"] = pos_df["Req_Field"].map(field_map)
pos_df["Tag"] = pos_df["Tag"].map(tag_map)

# pos_df = pos_df.sample(n=1000).reset_index(drop=True)

In [3]:
model = cp_model.CpModel()
solver = cp_model.CpSolver()

no_of_people = len(pers_df)
no_of_desk = len(pos_df)

assignments = {}
penalties   = {}

In [4]:
print(pers_df.head(5))

   Name_ID           Name  Citizenship  Tier  Field  Pass_Test1  Pass_Test2
0  P000001  Roxuqy Tiwoni            1     2      1        True       False
1  P000002    Lyxu Subyzu            3     1      6       False       False
2  P000003  Cagyla Gedyzo            1     2      2        True       False
3  P000004    Huwyle Wola            2     2      6        True       False
4  P000005    Levyjo Buti            1     2      1        True       False


In [ ]:
for i in range(no_of_people):
    for j in range(no_of_desk):
        pers = pers_df.loc[i]
        pos  = pos_df.loc[j]

        citizen_match = (pers["Citizenship"] == pos["Req_Citizenship"]) or (pos["Req_Citizenship"] == 4)
        field_match   = (pers["Field"] == pos["Req_Field"]) or (pos["Req_Field"] == 7)

        if citizen_match and field_match:
            assignments[i,j] = model.NewBoolVar(f"x_{i}_{j}")
            pers_tier = pers["Tier"]
            pos_tier  = pos["Req_Tier"]
            # setting max penalty 
            penalty_var = model.NewIntVar(0, 25, f"penalty_{i}_{j}") 
            penalties[i,j] = penalty_var
            
            if pos_tier == pers_tier:
                model.Add(penalty_var == 0).OnlyEnforceIf(assignments[i, j])

            elif pos_tier > pers_tier:
                tier_gap = pos_tier - pers_tier
                if tier_gap > 1:
                    # Too large a gap → cannot assign
                    model.Add(assignments[i, j] == 0)
                elif tier_gap == 1:
                    # One tier gap → test required
                    if pers_tier == tier_map["Junior"] and pos_tier == tier_map["Mid"]:
                        if pers["Pass_Test1"]:
                            model.Add(penalty_var == 4).OnlyEnforceIf(assignments[i, j])
                        else:
                            model.Add(assignments[i, j] == 0)
                            
                    elif pers_tier == tier_map["Mid"] and pos_tier == tier_map["Senior"]:
                        if pers["Pass_Test2"]:
                            model.Add(penalty_var == 4).OnlyEnforceIf(assignments[i, j])
                        else:
                            model.Add(assignments[i, j] == 0)
                        
            elif pos_tier < pers_tier:
                tier_gap = pers_tier - pos_tier
                if tier_gap > 1:
                    model.Add(assignments[i, j] == 0)
                else:
                    model.Add(penalty_var == 2).OnlyEnforceIf(assignments[i, j])

            role_penalty = 0
            if pos["Req_Citizenship"] == 4:
                role_penalty += 2  # higher penalty to avoid Any unless necessary
            if pos["Req_Field"] == 7:
                role_penalty += 2
            if role_penalty > 0:
                model.Add(penalty_var == role_penalty).OnlyEnforceIf(assignments[i, j])


In [ ]:
for i in range(no_of_people):
    feasible_jobs = [j for j in range(no_of_desk) if (i,j) in assignments]
    if feasible_jobs:
        model.Add(sum(assignments[i,j] for j in feasible_jobs) == 1)
    else:
        print(f"Warning: Person {pers_df.loc[i,'Name_ID']} has no feasible job!")
        
for j in range(no_of_desk): 
    feasible_people = [i for i in range(no_of_people) if (i,j) in assignments]
    if feasible_people:
        model.Add(sum(assignments[i,j] for i in feasible_people) <= 1)

In [ ]:
appointment_mismatch = model.NewIntVar(0, no_of_people * no_of_desk * 3, "appointment_mismatch")
penalty_sum_list = sum([penalties[i,j] * 5 for i,j in penalties]) #[penalties[i,j] * 5 for i,j in penalties]
model.Add(appointment_mismatch == penalty_sum_list)


In [ ]:
slack = model.NewIntVar(0, int(no_of_people*no_of_desk*3*0.05), "slack")
model.Maximize(sum(assignments.values()) - appointment_mismatch - slack)

In [ ]:
status_code = solver.Solve(model)
print(f"Relaxed solve: {solver.StatusName(status_code)} ({status_code})")
print("Objective value (relaxed):", solver.ObjectiveValue())
print(f"{solver.StatusName(status_code)} ({status_code})")
print("Best bound:", solver.BestObjectiveBound())


In [ ]:
inv_citizenship_map = {v: k for k, v in citizenship_map.items()}
inv_field_map = {v: k for k, v in field_map.items()}
inv_tag_map = {v: k for k, v in tag_map.items()}
inv_tier_map = {v: k for k, v in tier_map.items()}

pers_df["Citizenship"] = pers_df["Citizenship"].map(inv_citizenship_map)
pers_df["Tier"] = pers_df["Tier"].map(inv_tier_map)
pers_df["Field"] = pers_df["Field"].map(inv_field_map)

pos_df["Req_Citizenship"] = pos_df["Req_Citizenship"].map(inv_citizenship_map)
pos_df["Req_Tier"] = pos_df["Req_Tier"].map(inv_tier_map)
pos_df["Req_Field"] = pos_df["Req_Field"].map(inv_field_map)
pos_df["Tag"] = pos_df["Tag"].map(inv_tag_map)

In [ ]:
print(pos_df.head(5))
print(pers_df.head(5))

In [ ]:
print("Number of assignment variables:", len(assignments))

In [ ]:
assigned_rows = []
for (i,j), var in assignments.items():
    if solver.Value(var):
        assigned_rows.append({
            "Person_ID": pers_df.loc[i, "Name_ID"],
            "Person_Name": pers_df.loc[i, "Name"],
            "Pers_Cit": pers_df.loc[i, "Citizenship"],
            "Pers_Tier": pers_df.loc[i, "Tier"],
            "Pers_Field": pers_df.loc[i, "Field"],
            "Pass_1": pers_df.loc[i, "Pass_Test1"],
            "Pass_2": pers_df.loc[i, "Pass_Test2"],
            "Desk_ID": pos_df.loc[j, "Desk_ID"],
            "Req_Cit": pos_df.loc[j, "Req_Citizenship"],
            "Req_Tier": pos_df.loc[j, "Req_Tier"],
            "Req_Field": pos_df.loc[j, "Req_Field"],
            "Desk_Tag": pos_df.loc[j, "Tag"],
            "Tag_Level": pos_df.loc[j, "Level"],
        })


In [ ]:
assigned_df = pd.DataFrame(assigned_rows)
assigned_df.to_excel("new_update_assignment_results.xlsx", index=False)